# Train results — R2

**Strategy R2: protected repair (reject/rollback on regression)**

This notebook reproduces the `R2` column of the **Train results** tab in `Results.xlsx`, end to end. It:

1. Runs the repository's own analyzer (`scripts/analyze_repair_strategies.py`) on the `R2` training run (`experiments/subset_train-r2`).
2. Reads the analyzer's `overall_summary.csv` and `mechanism_summary.csv`.
3. Derives the **common-core** metrics (shared by every strategy) and the **R2-specific** rollback metrics.
4. Derives the final-outcome, repair-behaviour and efficiency analyses from the run, using the same code as `Train_results_R0.ipynb` so the two tables compare row for row, plus R2's rollback follow-up and cost.
5. Writes the exact grouped table (Category | Metric | R2 TRAIN result) to `Train_results_R2.xlsx`.

Everything is read-only analysis of an existing run: **no experiments, no model calls, no benchmark scoring.** The outputs below are produced by actually executing the cells.

## Parameters — edit these if your run lives elsewhere

`STRATEGY` is this notebook's strategy. `RUN_DIR` is the **training run folder** to analyze. If a new/rerun `subset_train` experiment is saved under a different folder name, change `RUN_DIR` below and re-run the whole notebook — the two summary CSVs and the output `.xlsx` will regenerate from that run.

In [1]:
# === EDIT THESE if your training run is in a different folder ===
STRATEGY = 'R2'
RUN_DIR  = 'experiments/subset_train-r2'   # the subset_train run this notebook analyzes
# ================================================================

## Step 0 — Setup and run the analyzer
We run from the repository root and point the analyzer at this strategy's training run. The analyzer writes its tables under a scratch output directory.

In [2]:
import subprocess, sys, os, csv, json
from pathlib import Path

# Resolve the repository root (the folder that contains 'scripts/' and 'experiments/').
ROOT = Path.cwd()
while not (ROOT / 'scripts' / 'analyze_repair_strategies.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'scripts' / 'analyze_repair_strategies.py').exists(), 'Run this from inside the repo.'
NB_DIR = Path.cwd()  # folder containing this notebook (before we chdir to repo root)
os.chdir(ROOT)
PY = sys.executable  # the kernel's interpreter (the project venv)
assert 'STRATEGY' in globals() and 'RUN_DIR' in globals(), 'Run the Parameters cell first.'

# Analyzer output is written here, NEXT TO THIS NOTEBOOK (not a temp dir),
# so you can open overall_summary.csv / mechanism_summary.csv yourself.
OUT_DIR = str(NB_DIR / 'analysis_output')

assert Path(RUN_DIR).is_dir(), f'Missing run folder: {RUN_DIR}'
print('Repo root :', ROOT)
print('Strategy  :', STRATEGY)
print('Run folder:', RUN_DIR)
print('Analyzer output dir:', OUT_DIR)

Repo root : C:\Users\Xue Jiarui\OneDrive - National University of Singapore\Documents\Agentic-Travel-Planner-Ablation-of-Verifications-
Strategy  : R2
Run folder: experiments/subset_train-r2
Analyzer output dir: C:\Users\Xue Jiarui\OneDrive - National University of Singapore\Documents\Agentic-Travel-Planner-Ablation-of-Verifications-\train_result_analysis\analysis_output


In [3]:
# Run the repo's analyzer for this one strategy. This only reads the saved run.
cmd = [PY, 'scripts/analyze_repair_strategies.py',
       '--run', f'{STRATEGY}={RUN_DIR}',
       '--output-dir', OUT_DIR, '--no-figures']
print('$', ' '.join(cmd))
res = subprocess.run(cmd, capture_output=True, text=True, encoding='utf-8',
                     env={**os.environ, 'PYTHONUTF8': '1'})
print(res.stdout.strip()[-1500:])
if res.returncode != 0:
    print('STDERR:', res.stderr[-2000:])
assert res.returncode == 0, 'analyzer failed'
print('\nAnalyzer exit code:', res.returncode)

$ C:\Users\Xue Jiarui\OneDrive - National University of Singapore\Documents\Agentic-Travel-Planner-Ablation-of-Verifications-\.venv\Scripts\python.exe scripts/analyze_repair_strategies.py --run R2=experiments/subset_train-r2 --output-dir C:\Users\Xue Jiarui\OneDrive - National University of Singapore\Documents\Agentic-Travel-Planner-Ablation-of-Verifications-\train_result_analysis\analysis_output --no-figures


R2: source=experiments\subset_train-r2, declared/evidenced queries=60, attempt queries=57, proposals=332, matrix cells=360

Analyzer exit code: 0


### Step 0b — Keep only the two summary CSVs
The analyzer emits many files we don't use (and an empty `figures/` folder). This removes everything except `overall_summary.csv` and `mechanism_summary.csv`, so re-running never leaves clutter behind.

In [4]:
# Keep the analyzer output tidy: the analyzer writes ~18 files (+ an empty
# figures/ dir) per strategy, but this notebook only needs the two summary
# CSVs. Delete everything else so the folder holds just what's useful.
import shutil
strat_out = Path(OUT_DIR) / STRATEGY
KEEP = {'overall_summary.csv', 'mechanism_summary.csv'}
removed = 0
for item in strat_out.iterdir():
    if item.name in KEEP:
        continue
    if item.is_dir():
        shutil.rmtree(item)
    else:
        item.unlink()
    removed += 1
print(f'Cleaned {strat_out}: removed {removed} extra item(s); kept {sorted(KEEP)}')
print('Files now:', sorted(p.name for p in strat_out.iterdir()))

Cleaned C:\Users\Xue Jiarui\OneDrive - National University of Singapore\Documents\Agentic-Travel-Planner-Ablation-of-Verifications-\train_result_analysis\analysis_output\R2: removed 16 extra item(s); kept ['mechanism_summary.csv', 'overall_summary.csv']
Files now: ['mechanism_summary.csv', 'overall_summary.csv']


## Step 1 — Load the analyzer's summary tables

> **Where do these CSVs come from?** They are **not** pre-existing files in the
> repo. **Step 0 above created them** by running the analyzer. They are written to
> `train_result_analysis/analysis_output/<STRATEGY>/` (printed by Step 0). If you
> look before running Step 0, the folder will not exist yet — that's expected.

`overall_summary.csv` holds counts/means per run; `mechanism_summary.csv` holds the strategy-specific mechanism metrics.

In [5]:
def load_summary(path):
    d = {}
    with open(path, encoding='utf-8') as f:
        for row in csv.DictReader(f):
            d[row['metric']] = row
    return d

overall = load_summary(f'{OUT_DIR}/{STRATEGY}/overall_summary.csv')
mech = load_summary(f'{OUT_DIR}/{STRATEGY}/mechanism_summary.csv')
print('overall_summary metrics:', len(overall))
print('mechanism_summary metrics:', len(mech))
print('\nmechanism metrics available:')
for k, v in mech.items():
    print(f'  {k}: count={v["count"]} valid_n={v["valid_n"]} rate={v["rate"]}')

overall_summary metrics: 85
mechanism_summary metrics: 5

mechanism metrics available:
  rejected: count=25 valid_n=332 rate=0.0753012048192771
  harmful_candidate_blocked: count=25 valid_n=332 rate=0.0753012048192771
  fixed_target_but_protected_regression_rejected: count=0 valid_n=332 rate=0
  rejected_and_continued: count=20 valid_n=332 rate=0.0602409638554217
  rejected_plan_differs_from_accepted: count=25 valid_n=25 rate=1


## Step 2 — Common-core metrics (shared by every strategy)
These come straight from `overall_summary.csv`. 'Confirmed unchanged attempts' is shown as `N (pct%)` of total attempts, matching the R1 template already in the spreadsheet.

In [6]:
def iv(metric):
    return int(float(overall[metric]['value']))

queries              = iv('declared_queries')
repair_attempts      = iv('total_attempts')
personal_attempts    = iv('personal_attempts')
commonsense_attempts = iv('commonsense_attempts')
unchanged            = iv('unchanged_confirmed')
unchanged_pct        = round(unchanged / repair_attempts * 100, 2)
personal_fixes       = iv('personal_fixed_count_sum')
personal_regressions = iv('personal_newly_violated_count_sum')

# (Category, Metric, Value). 'core' = metrics shared by every strategy.
core = [
    ('Dataset',          'Queries evaluated', queries),
    ('Repair behaviour', 'Total repair attempts', repair_attempts),
    ('Repair behaviour', 'Personal-constraint repair attempts', personal_attempts),
    ('Repair behaviour', 'Commonsense repair attempts', commonsense_attempts),
    ('Repair behaviour', 'Attempts that left the plan unchanged', f'{unchanged} ({unchanged_pct}%)'),
    ('Repair behaviour', 'Personal constraints fixed (across attempts)', personal_fixes),
    ('Repair behaviour', 'Personal constraints newly broken (across attempts)', personal_regressions),
]
for cat, k, v in core:
    print(f'  {cat:18} | {k:50} {v}')

  Dataset            | Queries evaluated                                  60
  Repair behaviour   | Total repair attempts                              332
  Repair behaviour   | Personal-constraint repair attempts                141
  Repair behaviour   | Commonsense repair attempts                        191
  Repair behaviour   | Attempts that left the plan unchanged              235 (70.78%)
  Repair behaviour   | Personal constraints fixed (across attempts)       2
  Repair behaviour   | Personal constraints newly broken (across attempts) 0


## Step 3 — R2-specific metrics
These describe R2's own mechanism, read from `mechanism_summary.csv`. Each is formatted `count / valid_n (rate%)`. Unlike R0's mechanism metrics, these **are** added to the table, under **Rollback (R2)**, because rollback is what distinguishes R2.

In [7]:
def fmt_count_rate(row):
    c = int(float(row['count'])); n = int(float(row['valid_n'])); r = float(row['rate']) * 100
    return f'{c} / {n} ({r:.1f}%)'

MECH_KEYS = [
    ('Harmful candidates blocked (rejected)', 'harmful_candidate_blocked'),
    ('Rejected: fixed target but caused a protected regression', 'fixed_target_but_protected_regression_rejected'),
    ('Rejected and loop continued', 'rejected_and_continued'),
    ('Rejected candidate differed from accepted plan', 'rejected_plan_differs_from_accepted'),
]
rollback = [('Rollback (R2)', label,
             fmt_count_rate(mech[metric]) if metric in mech else 'n/a (metric not produced for this run)')
            for label, metric in MECH_KEYS]
for cat, k, v in rollback:
    print(f'  {cat:14} | {k:58} {v}')

specific = []  # (category, metric, value) rows that WILL go below the spacer

  Rollback (R2)  | Harmful candidates blocked (rejected)                      25 / 332 (7.5%)
  Rollback (R2)  | Rejected: fixed target but caused a protected regression   0 / 332 (0.0%)
  Rollback (R2)  | Rejected and loop continued                                20 / 332 (6.0%)
  Rollback (R2)  | Rejected candidate differed from accepted plan             25 / 25 (100.0%)


In [8]:
# --- Load raw per-attempt and per-query data for analyses A-E ---
# (attempts.jsonl = one row per repair attempt; the per-query checkpoint JSON
#  carries initial_verification / final_verification and the repair_history.)
import json, glob
from collections import Counter

attempts = [json.loads(l) for l in open(Path(RUN_DIR, 'attempts.jsonl'), encoding='utf-8') if l.strip()]
ckpt_paths = sorted(glob.glob(str(Path(RUN_DIR, 'queries', '*', f'{STRATEGY}-*.json'))))
checkpoints = {}
for p in ckpt_paths:
    d = json.load(open(p, encoding='utf-8'))
    checkpoints[d['query_uid']] = d

def pc_pass_count(verif):
    # number of personal constraints that PASS (True) in a verification block
    pcs = (verif or {}).get('personal_constraints') or []
    return sum(1 for x in pcs if x is True)

print('attempts:', len(attempts), '| queries with checkpoints:', len(checkpoints))

# R2: attempts grouped per query in step order, for the rollback follow-up below.
from collections import defaultdict
by_query = defaultdict(list)
for a in attempts:
    by_query[a['query_uid']].append(a)
for steps in by_query.values():
    steps.sort(key=lambda a: a['step'])

attempts: 332 | queries with checkpoints: 60


### Final outcome — did the final plan beat the initial plan?
For each query, compare personal constraints passing in the **initial** vs **final** plan. Report how many queries improved, stayed the same, or worsened, and the net personal constraints gained (sum of per-query deltas).

In [9]:
improved = same = worsened = 0
delta_total = 0
for uid, d in checkpoints.items():
    pc_i = pc_pass_count(d.get('initial_verification'))
    pc_f = pc_pass_count(d.get('final_verification'))
    delta = pc_f - pc_i
    delta_total += delta
    if delta > 0: improved += 1
    elif delta < 0: worsened += 1
    else: same += 1
n_q = len(checkpoints)
def q(n): return f'{n}/{n_q} ({n / n_q * 100:.1f}%)'
outcome = [
    ('Final outcome', 'Queries improved', q(improved)),
    ('Final outcome', 'Queries unchanged', q(same)),
    ('Final outcome', 'Queries worsened', q(worsened)),
    ('Final outcome', 'Net personal constraints gained', delta_total),
]
specific += outcome
for cat, k, v in outcome: print(f'  {cat:14} | {k:40} {v}')

  Final outcome  | Queries improved                         4/60 (6.7%)
  Final outcome  | Queries unchanged                        55/60 (91.7%)
  Final outcome  | Queries worsened                         1/60 (1.7%)
  Final outcome  | Net personal constraints gained          5


### Final outcome — commonsense errors (start → end)
Counted from each query's saved result: the commonsense errors in `initial_verification` (before any repair) versus `final_verification` (the plan R2 ended with). Queries with no repair attempts have identical start and end counts.

In [10]:
assert len(checkpoints) == queries, f'expected {queries} result files, found {len(checkpoints)}'
cs_start = sum(len(d['initial_verification']['commonsense_errors']) for d in checkpoints.values())
cs_end = sum(len(d['final_verification']['commonsense_errors']) for d in checkpoints.values())
cs_change = (cs_end - cs_start) / cs_start * 100
cs_row = ('Final outcome', 'Commonsense errors (start \u2192 end)',
          f'{cs_start} \u2192 {cs_end} ({cs_change:+.1f}%)'.replace('-', '\u2212'))
specific.append(cs_row)
print(f'  {cs_row[0]:14} | {cs_row[1]:40} {cs_row[2]}')

  Final outcome  | Commonsense errors (start → end)         304 → 271 (−10.9%)


### Repair behaviour — what happens when R2 changes the plan?
Classify every attempt into mutually-exclusive outcomes using the measured slot change (`changed_slot_count`): confirmed no change (`0`), not measurable (`None`), or a confirmed change (`>0`) that either fixed ≥1 constraint without regression, introduced a regression (priority rule), or was neutral.

For R2 these rows describe the **candidate** each attempt proposed, including the rejected ones, so a rejected harmful candidate counts as 'introduced a regression' here even though R2 kept the previous plan.

In [11]:
conf_unchanged = fixed_only = regressed = other = 0
for a in attempts:
    csc = a.get('changed_slot_count')
    if csc == 0:
        conf_unchanged += 1
    elif isinstance(csc, (int, float)) and csc > 0:
        if a.get('newly_violated'): regressed += 1
        elif a.get('fixed'): fixed_only += 1
        else: other += 1
    else:
        other += 1  # change not measurable
N = len(attempts)
def pr(n): return f'{n}/{N} ({n / N * 100:.1f}%)'
behaviour = [
    ('Repair behaviour', 'Attempts with confirmed no change', pr(conf_unchanged)),
    ('Repair behaviour', 'Changed and fixed ≥1 without regression', pr(fixed_only)),
    ('Repair behaviour', 'Changed and introduced a regression', pr(regressed)),
]
specific += behaviour
for cat, k, v in behaviour: print(f'  {cat:16} | {k:45} {v}')

  Repair behaviour | Attempts with confirmed no change             214/332 (64.5%)
  Repair behaviour | Changed and fixed ≥1 without regression       1/332 (0.3%)
  Repair behaviour | Changed and introduced a regression           18/332 (5.4%)


### Repair by phase (context only — not added to the focused table)
Shown for discussion; these rows are intentionally left out of the final table to keep it concise.

In [12]:
def phase_stats(phase):
    rows = [a for a in attempts if a.get('repair_phase') == phase]
    n = len(rows)
    if n == 0: return n, 'n/a', 'n/a'
    fix = sum(1 for a in rows if a.get('fixed'))
    viol = sum(1 for a in rows if a.get('newly_violated'))
    f = lambda x: f'{x}/{n} ({x / n * 100:.1f}%)'
    return n, f(fix), f(viol)
for phase in ('personal', 'commonsense'):
    n, fix, viol = phase_stats(phase)
    print(f'  {phase:11} attempts={n}  fixed>=1={fix}  new-violation={viol}')

  personal    attempts=141  fixed>=1=2/141 (1.4%)  new-violation=0/141 (0.0%)
  commonsense attempts=191  fixed>=1=0/191 (0.0%)  new-violation=25/191 (13.1%)


### Rollback (R2) — what happens after a rejection?
Read from `attempts.jsonl`, since the analyzer does not report these.

- **Queries never accepted again after first rejection**: for each query with a rejection, every later attempt was also rejected.
- **Rejected candidates identical to an earlier rejected one**: the model proposed exactly the same rejected plan again for the same query.

In [13]:
rejected_queries = never_recovered = repeat_rejections = 0
for steps in by_query.values():
    rejected = [i for i, a in enumerate(steps) if not a['accepted']]
    if not rejected:
        continue
    rejected_queries += 1
    never_recovered += all(not a['accepted'] for a in steps[rejected[0]:])
    seen = []
    for i in rejected:
        repeat_rejections += steps[i]['candidate_plan'] in seen
        seen.append(steps[i]['candidate_plan'])
n_rejected = sum(not a['accepted'] for a in attempts)

def ratio(part, whole):
    return f'{part} / {whole} ({part / whole * 100:.1f}%)'

rollback += [
    ('Rollback (R2)', 'Queries never accepted again after first rejection', ratio(never_recovered, rejected_queries)),
    ('Rollback (R2)', 'Rejected candidates identical to an earlier rejected one', ratio(repeat_rejections, n_rejected)),
]
for cat, k, v in rollback[-2:]:
    print(f'  {cat:14} | {k:58} {v}')

  Rollback (R2)  | Queries never accepted again after first rejection         5 / 5 (100.0%)
  Rollback (R2)  | Rejected candidates identical to an earlier rejected one   6 / 25 (24.0%)


### Rollback cost (R2) — what do rejected candidates cost?
- **Tokens / time on rejected candidates**: the per-attempt `total_tokens_delta` and `attempt_time_seconds` of rejected attempts, out of all attempts. These per-attempt sums include verification time, so they differ slightly from the per-query averages under **Efficiency**.
- **Commonsense fixes discarded by rejections**: each query is replayed from its initial plan using its `repair_history`. For a rejected attempt, an error counts as discarded when it is in the kept plan but not in the rejected candidate (errors are matched by their exact message). The replay must end at each query's saved final errors, which checks it.

In [14]:
def total(key, rows):
    return sum(a.get(key) or 0 for a in rows)
rejected_rows = [a for a in attempts if not a['accepted']]
tok_rej, tok_all = total('total_tokens_delta', rejected_rows), total('total_tokens_delta', attempts)
sec_rej, sec_all = total('attempt_time_seconds', rejected_rows), total('attempt_time_seconds', attempts)

cs_discarded = 0
for uid, d in checkpoints.items():
    history = {h['step']: h for h in d['repair_history']}
    kept = Counter(d['initial_verification']['commonsense_errors'])
    for a in d['repair_attempts']:
        candidate = Counter(history[a['step']]['verification']['commonsense_errors'])
        if a['accepted']:
            kept = candidate
        else:
            cs_discarded += sum((kept - candidate).values())
    assert kept == Counter(d['final_verification']['commonsense_errors']), uid

rollback_cost = [
    ('Rollback cost (R2)', 'Tokens spent on rejected candidates',
     f'{tok_rej / 1e6:.2f}M / {tok_all / 1e6:.2f}M ({tok_rej / tok_all * 100:.1f}%)'),
    ('Rollback cost (R2)', 'Time spent on rejected candidates',
     f'{sec_rej:,.0f} s / {sec_all:,.0f} s ({sec_rej / sec_all * 100:.1f}%)'),
    ('Rollback cost (R2)', 'Commonsense fixes discarded by rejections', cs_discarded),
]
for cat, k, v in rollback_cost:
    print(f'  {cat:18} | {k:45} {v}')

  Rollback cost (R2) | Tokens spent on rejected candidates           0.89M / 5.63M (15.7%)
  Rollback cost (R2) | Time spent on rejected candidates             803 s / 6,261 s (12.8%)
  Rollback cost (R2) | Commonsense fixes discarded by rejections     64


### Efficiency — how expensive is R2?
Average tokens, model calls, and repair time **per query** (tokens averaged only over queries with complete token data). R2 regenerates the whole itinerary like R0, and also pays for candidates it then rejects.

In [15]:
from collections import defaultdict
per_q_tokens = defaultdict(int); per_q_time = defaultdict(float)
per_q_complete = defaultdict(lambda: True)
for a in attempts:
    uid = a['query_uid']
    tok = a.get('total_tokens_delta')
    if tok is None: per_q_complete[uid] = False
    else: per_q_tokens[uid] += tok
    per_q_time[uid] += a.get('repair_time_seconds') or 0.0
all_q = list({a['query_uid'] for a in attempts})
complete_q = [u for u in all_q if per_q_complete[u]]
def avg(d, keys): return (sum(d[k] for k in keys) / len(keys)) if keys else 0
efficiency = [
    ('Efficiency', 'Average tokens per query', round(avg(per_q_tokens, complete_q))),
    ('Efficiency', 'Average repair time per query', f'{avg(per_q_time, all_q):.1f} s'),
]
specific += efficiency
for cat, k, v in efficiency: print(f'  {cat:12} | {k:35} {v}')

  Efficiency   | Average tokens per query            99257
  Efficiency   | Average repair time per query       109.4 s


### Example rollback (context only — not added to the focused table)
A real attempt whose candidate broke a protected personal constraint and was rejected, printed for the write-up.

In [16]:
from chinatravel.evaluation.constraint_families import classify_constraint
def family_of(descriptors, idx):
    for d in descriptors or []:
        if d.get('constraint_index') == idx:
            try: fam = classify_constraint(d).get('family')
            except Exception: fam = None
            return fam or d.get('hard_logic_py', f'constraint {idx}')[:60]
    return f'constraint {idx}'
example = next((a for a in attempts if not a['accepted'] and a.get('newly_violated')), None)
if example is None:
    print('No candidate was rejected for a protected regression in this run.')
else:
    descs = example.get('constraint_descriptors') or []
    print(f'Query {example["query_uid"]}  (attempt step {example["step"]}, phase {example["repair_phase"]})')
    for i in example['newly_violated']:
        print(f'    constraint {i} [{family_of(descs, i)}]: Pass -> Fail in the candidate  (protected)')
    print(f'    decision: {example.get("decision_reason")}; the previous accepted plan was kept')

Query e20241028160440274536  (attempt step 1, phase commonsense)
    constraint 3 [monetary_budget]: Pass -> Fail in the candidate  (protected)
    decision: previously_satisfied_constraint_regressed; the previous accepted plan was kept


## Step 4 — Assemble the full table (Category | Metric | R2 TRAIN result)

In [17]:
# Header + all core rows, a spacer, then all strategy-specific rows.
# Each data row is (Category, Metric, Value).
# Same categories and order as R0, with R2's rollback rows before Efficiency.
specific = [r for r in specific if r[0] != 'Efficiency'] + rollback + rollback_cost + efficiency
HEADER = ('Category', 'Metric', f'{STRATEGY} TRAIN result')
rows = [HEADER]
rows += core
rows += [('', '', '')]  # spacer: ABOVE = common to every strategy; BELOW = R2-specific
rows += specific

for cat, k, v in rows:
    print(f'{cat:18} | {k:60} | {v}')

Category           | Metric                                                       | R2 TRAIN result
Dataset            | Queries evaluated                                            | 60
Repair behaviour   | Total repair attempts                                        | 332
Repair behaviour   | Personal-constraint repair attempts                          | 141
Repair behaviour   | Commonsense repair attempts                                  | 191
Repair behaviour   | Attempts that left the plan unchanged                        | 235 (70.78%)
Repair behaviour   | Personal constraints fixed (across attempts)                 | 2
Repair behaviour   | Personal constraints newly broken (across attempts)          | 0
                   |                                                              | 
Final outcome      | Queries improved                                             | 4/60 (6.7%)
Final outcome      | Queries unchanged                                            | 55/60 (91.7%)
F

## Step 5 — Write the exact table to `Train_results_R2.xlsx`

In [18]:
import openpyxl
from openpyxl.styles import Font, Alignment

wb = openpyxl.Workbook()
ws = wb.active
ws.title = f'{STRATEGY} train results'

for r, (cat, k, v) in enumerate(rows, start=1):
    ws.cell(r, 1, cat)
    ws.cell(r, 2, k)
    vcell = ws.cell(r, 3, v)
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        vcell.number_format = '#,##0'  # show large numbers with thousands separators

# bold header row
for col in (1, 2, 3):
    ws.cell(1, col).font = Font(bold=True)
# wrap long metric text
for r in range(1, len(rows) + 1):
    ws.cell(r, 2).alignment = Alignment(wrap_text=True, vertical='top')
ws.column_dimensions['A'].width = 20
ws.column_dimensions['B'].width = 60
ws.column_dimensions['C'].width = 24

out_xlsx = NB_DIR / f'Train_results_{STRATEGY}.xlsx'
wb.save(out_xlsx)
print('Wrote', out_xlsx)
print('Rows written:', len(rows))

Wrote C:\Users\Xue Jiarui\OneDrive - National University of Singapore\Documents\Agentic-Travel-Planner-Ablation-of-Verifications-\train_result_analysis\Train_results_R2.xlsx
Rows written: 28


## Step 6 — Read the file back to confirm it saved correctly

In [19]:
chk = openpyxl.load_workbook(out_xlsx, data_only=True)
cs = chk.active
print('Sheet:', cs.title, '| dims:', cs.dimensions)
print()
for row in cs.iter_rows(values_only=True):
    if any(c not in (None, '') for c in row):
        cat, k, v = (row + (None, None, None))[:3]
        print(f'{str(cat or ""):16} | {str(k or ""):58} | {v}')

Sheet: R2 train results | dims: A1:C28

Category         | Metric                                                     | R2 TRAIN result
Dataset          | Queries evaluated                                          | 60
Repair behaviour | Total repair attempts                                      | 332
Repair behaviour | Personal-constraint repair attempts                        | 141
Repair behaviour | Commonsense repair attempts                                | 191
Repair behaviour | Attempts that left the plan unchanged                      | 235 (70.78%)
Repair behaviour | Personal constraints fixed (across attempts)               | 2
Repair behaviour | Personal constraints newly broken (across attempts)        | 0
Final outcome    | Queries improved                                           | 4/60 (6.7%)
Final outcome    | Queries unchanged                                          | 55/60 (91.7%)
Final outcome    | Queries worsened                                           | 1/60 (